In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass

# =========================
# MODEL DATA
# =========================
@dataclass
class Order:
    customer_name: str
    total_price: float
    status: str = "open"


# =========================
# ABSTRAKSI (INTERFACE)
# =========================
class IPaymentProcessor(ABC):
    @abstractmethod
    def process(self, order: Order) -> bool:
        pass


class INotificationService(ABC):
    @abstractmethod
    def send(self, order: Order):
        pass


# =========================
# IMPLEMENTASI PAYMENT
# =========================
class CreditCardProcessor(IPaymentProcessor):
    def process(self, order: Order) -> bool:
        print(f"Payment: Memproses Kartu Kredit untuk {order.customer_name}")
        return True


class BankTransferProcessor(IPaymentProcessor):
    def process(self, order: Order) -> bool:
        print(f"Payment: Memproses Bank Transfer untuk {order.customer_name}")
        return True


class QrisProcessor(IPaymentProcessor):
    def process(self, order: Order) -> bool:
        print(f"Payment: Memproses QRIS untuk {order.customer_name}")
        return True


# =========================
# IMPLEMENTASI NOTIFIKASI
# =========================
class EmailNotifier(INotificationService):
    def send(self, order: Order):
        print(f"Notif: Email dikirim ke {order.customer_name}")


# =========================
# SERVICE (SRP + DIP)
# =========================
class CheckoutService:
    def __init__(self, payment_processor: IPaymentProcessor, notifier: INotificationService):
        self.payment_processor = payment_processor
        self.notifier = notifier

    def checkout(self, order: Order) -> bool:
        print(f"\nMemulai checkout untuk {order.customer_name}...")

        payment_success = self.payment_processor.process(order)

        if not payment_success:
            print("Checkout gagal.")
            return False

        order.status = "paid"
        self.notifier.send(order)
        print("Checkout berhasil.\n")
        return True


# =========================
# PROGRAM UTAMA
# =========================
def main():
    email_service = EmailNotifier()

    # Skenario 1: Credit Card
    order1 = Order("Andi", 500000)
    cc_processor = CreditCardProcessor()
    checkout1 = CheckoutService(cc_processor, email_service)
    checkout1.checkout(order1)

    # Skenario 2: QRIS (bukti OCP, tanpa ubah kode lama)
    order2 = Order("Budi", 100000)
    qris_processor = QrisProcessor()
    checkout2 = CheckoutService(qris_processor, email_service)
    checkout2.checkout(order2)

    # Skenario 3: Bank Transfer
    order3 = Order("Siti", 250000)
    bank_processor = BankTransferProcessor()
    checkout3 = CheckoutService(bank_processor, email_service)
    checkout3.checkout(order3)


if __name__ == "__main__":
    main()